In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import time 

def fetch_article_urls(base_url, page_number):
    """
    Fetch article URLs from a given page.
    """
    article_urls = set()
    url = f"{base_url}/page/{page_number}/"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for link in soup.find_all("a", href=True):
            href = urljoin(url, link['href'])
            if re.match(r'https://24horaspuebla.com/\d{4}/\d{2}/.*/', href):
                article_urls.add(href)
    except requests.RequestException as e:
        print(f"Error fetching URLs from {url}: {e}")
    return list(article_urls)

def clean_main_text(text):
    """
    Remove unwanted phrases from the main text.
    """
    unwanted_phrases = [
        "Diario 24 Horas Puebla \n\t\t\t\t\tEl diario sin límites\t\t\t\t"
    ]
    for phrase in unwanted_phrases:
        text = text.replace(phrase, "")
    return text.strip()

def extract_article_details(url):
    """
    Extract title, main text, author, and date from an article.
    """

    
    article_data = {}
    try:
        article_data['url'] = url
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title_element = soup.find("h1", class_="entry-title")
        article_data['title'] = title_element.text.strip() if title_element else None

        # Extract and clean main text
        paragraphs = [p.get_text() for p in soup.find_all('p')]
        raw_text = ' '.join(paragraphs)
        article_data['main_text'] = clean_main_text(raw_text)

        # Extract author
        author_element = soup.find("span", class_="author vcard")
        article_data['author'] = author_element.text.strip() if author_element else None

        # Extract date
        date_element = soup.find("time")
        article_data['date'] = date_element['datetime'] if date_element and 'datetime' in date_element.attrs else None

    except requests.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return None

    return article_data

def scrape_articles(base_url, start_page=1, end_page=10):
    """
    Scrape articles from a given range of pages and save results after every 100 pages.
    """
    df = pd.DataFrame()
    all_articles = []
    for page_number in range(start_page, end_page + 1):
        print(f"Extracting articles from page {page_number}")
        urls = fetch_article_urls(base_url, page_number)
        for url in urls:
            article = extract_article_details(url)
            if article:
                all_articles.append(article)
            
        
        df_new = pd.DataFrame(all_articles)
        df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes
        df = pd.concat([df, df_new])
        all_articles = []
        # Save progress every 100 pages or at the end of the scraping session
        if (page_number % 50 == 0 or page_number == end_page):
            df = df.drop_duplicates(['url']).reset_index(drop=True)
            df.to_parquet(f"../../data/00-newspaper_data/crawler/horas24puebla/articles.parquet", index=False)
            print(f"Data saved'")
                
    
    return(df)

        

# Define the base URL
base_url = "https://24horaspuebla.com/minuto-a-minuto"
# Example usage: scrape and save the first 10 pages
sc = scrape_articles(base_url, 1, 12710)


Extracting articles from page 1
Extracting articles from page 2
Extracting articles from page 3
Extracting articles from page 4
Extracting articles from page 5
Extracting articles from page 6
Extracting articles from page 7
Extracting articles from page 8
Extracting articles from page 9
Extracting articles from page 10
Extracting articles from page 11
Extracting articles from page 12
Extracting articles from page 13
Extracting articles from page 14
Extracting articles from page 15
Extracting articles from page 16
Extracting articles from page 17
Extracting articles from page 18
Extracting articles from page 19
Extracting articles from page 20
Extracting articles from page 21
Extracting articles from page 22
Extracting articles from page 23
Extracting articles from page 24
Extracting articles from page 25
Extracting articles from page 26
Extracting articles from page 27
Extracting articles from page 28
Extracting articles from page 29
Extracting articles from page 30
Extracting articles

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_38449/1680021987.py:89: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 2139
Extracting articles from page 2140
Extracting articles from page 2141
Extracting articles from page 2142
Extracting articles from page 2143
Extracting articles from page 2144
Extracting articles from page 2145
Extracting articles from page 2146
Extracting articles from page 2147
Extracting articles from page 2148
Extracting articles from page 2149
Extracting articles from page 2150
Data saved'
Extracting articles from page 2151
Extracting articles from page 2152
Extracting articles from page 2153
Extracting articles from page 2154
Extracting articles from page 2155
Extracting articles from page 2156
Extracting articles from page 2157
Extracting articles from page 2158
Extracting articles from page 2159
Extracting articles from page 2160
Extracting articles from page 2161
Extracting articles from page 2162
Extracting articles from page 2163
Extracting articles from page 2164
Extracting articles from page 2165
Extracting articles from page 2166
Extracti

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_38449/1680021987.py:89: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 2403
Extracting articles from page 2404
Extracting articles from page 2405
Extracting articles from page 2406
Extracting articles from page 2407
Extracting articles from page 2408
Extracting articles from page 2409
Extracting articles from page 2410
Extracting articles from page 2411
Extracting articles from page 2412
Extracting articles from page 2413
Extracting articles from page 2414
Extracting articles from page 2415
Extracting articles from page 2416
Extracting articles from page 2417
Extracting articles from page 2418
Extracting articles from page 2419
Extracting articles from page 2420
Extracting articles from page 2421
Extracting articles from page 2422
Extracting articles from page 2423
Extracting articles from page 2424
Extracting articles from page 2425
Extracting articles from page 2426
Extracting articles from page 2427
Extracting articles from page 2428
Extracting articles from page 2429
Extracting articles from page 2430
Extracting articles 

...

Extracting articles from page 6657
Extracting articles from page 6658
Extracting articles from page 6659
Extracting articles from page 6660
Extracting articles from page 6661
Extracting articles from page 6662
Extracting articles from page 6663
Extracting articles from page 6664
Extracting articles from page 6665
Extracting articles from page 6666
Extracting articles from page 6667
Extracting articles from page 6668
Extracting articles from page 6669
Extracting articles from page 6670
Extracting articles from page 6671
Extracting articles from page 6672
Extracting articles from page 6673
Extracting articles from page 6674
Extracting articles from page 6675
Extracting articles from page 6676
Extracting articles from page 6677
Extracting articles from page 6678
Extracting articles from page 6679
Extracting articles from page 6680
Extracting articles from page 6681
Extracting articles from page 6682
Extracting articles from page 6683
Extracting articles from page 6684
Extracting articles 

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_38449/1680021987.py:89: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 6892
Extracting articles from page 6893
Extracting articles from page 6894
Extracting articles from page 6895
Extracting articles from page 6896
Extracting articles from page 6897
Extracting articles from page 6898
Extracting articles from page 6899
Extracting articles from page 6900
Data saved'
Extracting articles from page 6901
Extracting articles from page 6902
Extracting articles from page 6903
Extracting articles from page 6904
Extracting articles from page 6905
Extracting articles from page 6906
Extracting articles from page 6907
Extracting articles from page 6908
Extracting articles from page 6909
Extracting articles from page 6910
Extracting articles from page 6911
Extracting articles from page 6912
Extracting articles from page 6913
Extracting articles from page 6914
Extracting articles from page 6915
Extracting articles from page 6916
Extracting articles from page 6917
Extracting articles from page 6918
Extracting articles from page 6919
Extracti

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_38449/1680021987.py:89: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 11330
Extracting articles from page 11331
Extracting articles from page 11332
Extracting articles from page 11333
Extracting articles from page 11334
Extracting articles from page 11335
Extracting articles from page 11336
Extracting articles from page 11337
Extracting articles from page 11338
Extracting articles from page 11339
Extracting articles from page 11340
Extracting articles from page 11341
Extracting articles from page 11342
Extracting articles from page 11343
Extracting articles from page 11344
Extracting articles from page 11345
Extracting articles from page 11346
Extracting articles from page 11347
Extracting articles from page 11348
Extracting articles from page 11349
Extracting articles from page 11350
Data saved'
Extracting articles from page 11351
Extracting articles from page 11352
Extracting articles from page 11353
Extracting articles from page 11354
Extracting articles from page 11355
Extracting articles from page 11356
Extracting artic